# 08.6 - GPT-style Language Models

**Phase:** 08 - Transformers

**Status:** VERIFIED

---

## 1. What Are We Solving?

GPT is a **decoder-only** transformer. It learns to predict the *next character* given the past, which lets it **generate** text. Every GPT conversation you have is a loop of: read the context, predict the next token, append it, repeat.

## 2. Why Does This Matter?

Next-token prediction is the single objective behind ChatGPT and every modern chat model. If you understand teacher-forced training plus sampling (temperature, top-k, top-p), you understand how generation actually works.

## 3. Prerequisites

- Causal masking in the decoder (08.4) and transformer blocks (08.1)
- Cross-entropy training basics

## 4. Learning Objectives

By the end of this unit, you should:
- Build a tiny decoder-only transformer and train it on next-character prediction
- Read the probabilities it learned for continuations
- Implement temperature, top-k and top-p sampling by hand
- Show why greedy decoding degenerates into loops and how sampling fixes it

## 5. Mental Model

The trained model is a **probability machine**: given any prefix, it outputs a whole distribution over the next character. A generation *policy* (greedy / temperature / top-k / top-p) decides which character to actually pick.

```text
prefix 'th' -> P(next = 'e') = 0.99, others ~0   (a language model)
greedy picks the argmax. top-k samples among the k biggest. top-p samples the nucleus.
```


## 6. Setup


In [1]:
import matplotlib
matplotlib.use('Agg')
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)
print('torch', torch.__version__)


torch 2.13.0+cpu


## 7. The Synthetic Language

We hand-build a tiny 'grammar' over characters (space + a..t). The next character depends on the previous one (**bigram transitions**), with a few deterministic transitions and a few choices. The model's job is to discover these transitions.
```
' ' -> t(0.6) c(0.2) m(0.2)
t -> h(1.0)     h -> e(1.0)    e -> ' '(1.0)     (word 'the')
c -> a(1.0)     a -> t(1.0)                       (word 'cat')
m -> a(0.5) ' '(0.5)                              ('ma' / 'm ')
r -> ' '(0.5) a(0.5)                              ('r ' / 'ra')
```
So 'the cat the cat ...' style text, where greedy decoding will loop on 'the'.


In [2]:
CHARS = [' ', 't', 'h', 'e', 'c', 'a', 'm', 'r']
ch2i = {c: i for i, c in enumerate(CHARS)}
V, TXT_LEN = len(CHARS), 16

trans = {
    ' ':  [('t', 0.6), ('c', 0.2), ('m', 0.2)],
    't':  [('h', 1.0)],
    'h':  [('e', 1.0)],
    'e':  [(' ', 1.0)],
    'c':  [('a', 1.0)],
    'a':  [('t', 1.0)],
    'm':  [('a', 0.5), (' ', 0.5)],
    'r':  [(' ', 0.5), ('a', 0.5)],
}
P_true = torch.full((V, V), 1e-9)   # smoothing so every transition is possible
for prev, nxts in trans.items():
    for nxt, p in nxts:
        P_true[ch2i[prev], ch2i[nxt]] = p
P_true = P_true / P_true.sum(1, keepdim=True)

def sample_corpus(chars):
    out = [' ']
    for _ in range(chars):
        out.append(CHARS[torch.multinomial(P_true[ch2i[out[-1]]], 1).item()])
    return ''.join(out)

corpus = sample_corpus(3000)
print('First 120 chars of the corpus:')
print(repr(corpus[:120]))
print('P(next | t):', (P_true[ch2i['t']] * 100).round().tolist())


First 120 chars of the corpus:
' the the the the the the the the the the the cathe m the the mathe cathe the m cathe the the mathe m the mathe the cathe'
P(next | t): [0.0, 0.0, 100.0, 0.0, 0.0, 0.0, 0.0, 0.0]


## 8. The Tiny GPT Model

Decoder-only: masked self-attention + feed-forward, plus an LM head mapping hidden states back to vocab logits. We train with **teacher forcing**: the input is shifted by one, and at every position the model must predict the *next* character.


In [3]:
def causal_mask(n):
    return torch.tril(torch.ones(n, n, dtype=torch.bool))

class MultiHeadAttention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dk = h, d // h
        self.wq, self.wk, self.wv, self.wo = (nn.Linear(d, d) for _ in range(4))
    def forward(self, x, mask):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.h, self.dk).transpose(1, 2)
        k = self.wk(x).view(B, T, self.h, self.dk).transpose(1, 2)
        v = self.wv(x).view(B, T, self.h, self.dk).transpose(1, 2)
        s = q @ k.transpose(-2, -1) / math.sqrt(self.dk)
        s = s.masked_fill(~mask[None, None], float('-inf'))
        w = F.softmax(s, dim=-1)
        o = (w @ v).transpose(1, 2).reshape(B, T, -1)
        return self.wo(o)

class DecoderBlock(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.attn = MultiHeadAttention(d, h)
        self.n1 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.n2 = nn.LayerNorm(d)
    def forward(self, x, mask):
        a = self.attn(x, mask)
        x = self.n1(x + a)
        return self.n2(x + self.ff(x))

class TinyGPT(nn.Module):
    def __init__(self, vocab, d=32, h=4, n_layers=2, max_len=64):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.pos = nn.Parameter(torch.randn(1, max_len, d) * 0.02)
        self.blocks = nn.ModuleList([DecoderBlock(d, h) for _ in range(n_layers)])
        self.lm = nn.Linear(d, vocab)
    def forward(self, x):
        n = x.size(1)
        h = self.emb(x) + self.pos[:, :n]
        for b in self.blocks:
            h = b(h, causal_mask(n)[:n, :n])
        return self.lm(h)

gpt = TinyGPT(V)
logits = gpt(torch.randint(0, V, (2, TXT_LEN)))
print('logits shape:', tuple(logits.shape))
n = sum(p.numel() for p in gpt.parameters())
print('TinyGPT parameters:', f'{n:,}')


logits shape: (2, 16, 8)
TinyGPT parameters: 27,976


## 9. Train: Next-Character Prediction

We convert the character corpus into (input, target) pairs by shifting by one, then train with cross-entropy on every position in parallel (teacher forcing).


In [4]:
ids = torch.tensor([ch2i[c] for c in corpus])

def batches_from(corpus_ids, blen, bs):
    n = (len(corpus_ids) - 1) // (blen)
    idx = torch.randperm(n)[:bs]
    starts = idx * blen
    x = torch.stack([corpus_ids[s:s + blen] for s in starts])
    y = torch.stack([corpus_ids[s + 1:s + 1 + blen] for s in starts])
    return x, y

opt = torch.optim.Adam(gpt.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

for step in range(1, 251):
    x, y = batches_from(ids, TXT_LEN, 32)
    opt.zero_grad()
    loss = loss_fn(gpt(x).reshape(-1, V), y.reshape(-1))
    loss.backward()
    opt.step()
    if step % 100 == 0:
        print(f'step {step:4d}: loss={loss.item():.3f}  bpc={loss.item()/math.log(2):.3f}')


step  100: loss=0.254  bpc=0.366


step  200: loss=0.243  bpc=0.351


## 10. What Did It Learn? Check the Probabilities

Feed a one-character context and print the full distribution the model assigns to the next character. It should match the true bigram transitions.


In [5]:
def next_dist(ch):
    with torch.no_grad():
        logits = gpt(torch.tensor([[ch2i[ch]]]))
        return F.softmax(logits[0, 0], dim=-1)

for ch in ['t', ' ', 'm']:
    p = next_dist(ch)
    top = p.topk(4)
    s = ', '.join(f'{CHARS[i]}:{v:.2f}' for v, i in zip(top.values, top.indices))
    print(f'P(next | {ch!r}) = {s}')

true_t = P_true[ch2i[' ']]
learned_t = next_dist(' ')
print('\nKL(true || learned) from space:',
      F.kl_div(learned_t.log(), true_t, reduction='batchmean').item())


P(next | 't') = h:1.00, t:0.00, c:0.00, e:0.00
P(next | ' ') = t:0.69, m:0.24, c:0.06, r:0.00
P(next | 'm') =  :0.80, a:0.20, r:0.00, e:0.00



KL(true || learned) from space: 0.013372503221035004


## 11. Sampling Toolbox: Temperature, Top-k, Top-p

Generation applies three classic tricks to the logits before picking a character:
- **Temperature**: divide logits by T. T<1 sharpens (more greedy), T>1 flattens (more random).
- **Top-k**: keep only the k largest logits, zero the rest.
- **Top-p (nucleus)**: keep the smallest set of tokens whose cumulative probability reaches p.


In [6]:
def temperature(logits, T=1.0):
    return logits / T

def top_k(logits, k):
    v, _ = torch.topk(logits, k)
    floor = v[-1]
    masked = logits.clone()
    masked[logits < floor] = float('-inf')
    return masked

def top_p(logits, p=0.9):
    probs = F.softmax(logits, dim=-1)
    sp, si = probs.sort(descending=True)
    cum = sp.cumsum(-1)
    keep = cum <= p
    keep[..., 1:] = keep[..., 1:] | (cum[..., :-1] <= p)
    keep[..., 0] = True    # the most probable token always survives, even if > p
    masked = logits.clone()
    masked.masked_fill_(~keep.gather(-1, si.argsort(-1)), float('-inf'))
    return masked

def sample_policy(logits, T=1.0, k=None, p=None):
    lg = temperature(logits, T)
    if k is not None:
        lg = top_k(lg, k)
    if p is not None:
        lg = top_p(lg, p)
    return torch.multinomial(F.softmax(lg, dim=-1), 1).item()

with torch.no_grad():
    base = gpt(torch.tensor([[ch2i['t']]]))[0, 0]
for name, lg in [('raw', base), ('T=0.3', temperature(base, 0.3)),
                 ('T=3.0', temperature(base, 3.0)), ('top-k=2', top_k(base, 2)),
                 ('top-p=0.7', top_p(base, 0.7))]:
    p = F.softmax(lg, dim=-1)
    print(f'{name:9s}: h={p[ch2i["h"]]:.3f} e={p[ch2i["e"]]:.3f} a={p[ch2i["a"]]:.3f} '
          f't={p[ch2i["t"]]:.3f}  (max={p.max():.2f})')


raw      : h=0.997 e=0.000 a=0.000 t=0.001  (max=1.00)
T=0.3    : h=1.000 e=0.000 a=0.000 t=0.000  (max=1.00)
T=3.0    : h=0.656 e=0.050 a=0.048 t=0.061  (max=0.66)
top-k=2  : h=0.999 e=0.000 a=0.000 t=0.001  (max=1.00)
top-p=0.7: h=1.000 e=0.000 a=0.000 t=0.000  (max=1.00)


## 12. Failure Case: Greedy Decoding Loops

If we always take the argmax ('the' -> 'the' -> 'the'), the model generates a perfect but empty loop. This is the classic degeneration error that sampling was invented to fix.


In [7]:
def generate(policy, n=60, start=' '):
    ctx = [ch2i[ch] for ch in start]
    out = start
    for _ in range(n):
        with torch.no_grad():
            logits = gpt(torch.tensor([ctx[-4:]]) )   # sliding window of context
        nxt = policy(logits[0, -1])
        out += CHARS[nxt]
        ctx.append(nxt)
    return out

def greedy(L):
    return L.argmax().item()

print('GREEDY:')
print(generate(greedy, 70))
print('\nTEMPERATURE 0.9:')
torch.manual_seed(1)
print(generate(lambda L: sample_policy(L, T=0.9), 70))


GREEDY:


 the the the the the the the the the the the the the the the the the th

TEMPERATURE 0.9:


 the m the the the the the the the the the the the the the the the math


## 13. Sampling Breaks the Loop (and its Diversity)

Greedy uses one bigram ('th') repeatedly. Samplers explore the grammar's choices. We measure diversity with distinct bigrams in a long sample.


In [8]:
def diversity(text, n=2):
    return len({text[i:i + n] for i in range(len(text) - n)})

torch.manual_seed(3);  g1 = generate(greedy, 120)
torch.manual_seed(3);  g2 = generate(lambda L: sample_policy(L, T=0.9), 120)
torch.manual_seed(3);  g3 = generate(lambda L: sample_policy(L, T=1.0, k=3), 120)
torch.manual_seed(3);  g4 = generate(lambda L: sample_policy(L, T=1.0, p=0.8), 120)
for name, g in [('greedy ', g1), ('temp0.9', g2), ('top-k=3', g3), ('top-p=.8', g4)]:
    print(f'{name}: bigrams={diversity(g):3d}  {g[:44]!r}')
print('\nGreedy repeats one loop; sampling replaces it with probability-driven variety.')
print('Top-p=0.8 keeps 80% of the probability mass - here that quietly drops the'
      ' 20% mass third word, so it stays more conservative than top-k=3.')


greedy : bigrams=  4  ' the the the the the the the the the the the'
temp0.9: bigrams= 10  ' the the mathe cathe m the the m the the the'
top-k=3: bigrams= 10  ' the the mathe cathe m the the m the the the'
top-p=.8: bigrams=  8  ' the the mathe m the m the the m the the the'

Greedy repeats one loop; sampling replaces it with probability-driven variety.
Top-p=0.8 keeps 80% of the probability mass - here that quietly drops the 20% mass third word, so it stays more conservative than top-k=3.


## 14. Debugging

| Symptom | Possible Cause | Verify | Fix |
|---|---|---|---|
| Loss stuck near ln(V) | Nothing learned | Check learned dist vs true | More steps / LR |
| Samples are gibberish | Context window too short | Generate with 4+ chars | Larger window |
| Samples keep looping | Always sampling argmax | Check policy used | Use T/k/p sampling |
| Output repeats a phrase | Tempp low + fluent model | Inspect distribution sharpness | Raise T or top-p |

## 15. Real-World Considerations

- Real models learn much longer dependencies; bigram context here is a toy.
- Greedy 4-char window in `generate` mimics KV-cache-free autoregression; real engines cache keys/values.
- Sampling without temperature control causes repetition (ChatGPT-style 'helpful' outputs use tuned T and no-repeat penalties).

## 16. Common Mistakes

- Letting the model see the future during training (no causal mask) - it then fails at generation.
- Evaluating loss but never tasting the sampled text - loss tells only part of the story.
- Using temperature > 1 everywhere (too random) or < 0.3 everywhere (copy-paste).

## 17. When NOT to Use

- Tasks that only need understanding (classification): use an encoder or encoder-decoder.
- Deterministic outputs (autocomplete of code): consider stricter constraints than sampling.

## 18. Challenge

Find the temperatures where the model breaks. Generate text at T=10 and at T=0.01 and compare: one should be white noise over characters, the other an almost exact repeat.


In [9]:
torch.manual_seed(5);  g_hot = generate(lambda L: sample_policy(L, T=10.0), 60)
torch.manual_seed(5);  g_cold = generate(lambda L: sample_policy(L, T=0.01), 60)
print('T=10.0 (too hot):', repr(g_hot))
print('T=0.01 (too cold):', repr(g_cold))
print('\nHigh temperature approaches uniform random; low temperature approaches greedy.')


T=10.0 (too hot): ' tmmeertr mamrhe emcehherthtcaacac ta thae crhetthcetc maa te'
T=0.01 (too cold): ' the the the the the the the the the the the the the the the '

High temperature approaches uniform random; low temperature approaches greedy.


## 19. Closed-Book Recall

1. Why is teacher forcing needed, and what does the model not experience during training?
2. What does temperature literally do to the logits?
3. Why does greedy decoding loop on a fluent model?
4. Top-k vs top-p: what differs about how each decides the candidate set?

## 20. Teach-Back Questions

Explain to another person:
- The generate loop: context -> probs -> pick -> append.
- The role of the causal mask in making next-token prediction honest.
- When you would reach for top-p versus top-k.

## 21. Summary

You built a tiny GPT, trained it on a synthetic bigram language with teacher forcing, inspected the probability distributions it learned, implemented temperature / top-k / top-p sampling from scratch, and showed how greedy decoding degenerates into loops while sampling produces diverse, grammatical text.

## 22. Further Experiment

- Train on a longer synthetic context: make transitions depend on the previous TWO characters and increase the window.
- Re-run the sampling comparison with the sliding window reduced to 1 char - does politeness of generated text survive?

## 23. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, torch
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
